# Hamilton Heater Shaker

The Hamilton Heater Shaker is a combined temperature-control and orbital-shaking device for microplates. It supports:

- [Shaking](../../capabilities/shaking) (orbital, 20--2000 rpm)
- [Temperature control](../../capabilities/temperature-control) (heating only, RT+5 °C to 105 °C)
- Plate locking

Active cooling is **not** supported.

| Variant (Cat. No.) | Shaking Orbit | Shaking Speed | Max. Loading |
|---|---|---|---|
| 199027 | 1.5 mm | 100--1800 rpm | -- |
| 199033 | 2.0 mm | 100--2500 rpm | -- |
| 199034 | 3.0 mm | 100--2400 rpm | 500 g |

All variants share the same driver and are represented by the `HamiltonHeaterShaker` class.

The Heater Shaker can be controlled through two interfaces:

- **HamiltonHeaterShakerBox** -- a USB control box supporting up to 8 heater shakers. One heater shaker is connected via USB to the host computer and acts as a gateway.
- **Hamilton STAR** -- the heater shaker plugs directly into one of the two RS232 ports on the STAR liquid handler.

## Setup with the control box

In [ ]:
from pylabrobot.hamilton.heater_shaker import HamiltonHeaterShaker, HamiltonHeaterShakerBox

box = HamiltonHeaterShakerBox()
await box.setup()

hs = HamiltonHeaterShaker(name="hhs", index=0, interface=box)
await hs.setup()

The `index` parameter identifies which heater shaker on the box you are addressing. Set `index=0` for the unit that is connected via USB to the host computer. Other units on the same box use `index=1`, `index=2`, etc. (requires setting the DIP switch on the bottom of each module).

## Alternative: setup via a Hamilton STAR

If the heater shaker is plugged directly into a STAR liquid handler, use the STAR backend as the interface. The back RS232 port corresponds to `index=1` and the front port to `index=2`.

In [ ]:
from pylabrobot.hamilton.heater_shaker import HamiltonHeaterShaker
from pylabrobot.liquid_handling import LiquidHandler, STARBackend
from pylabrobot.resources import STARDeck

star_backend = STARBackend()
lh = LiquidHandler(backend=star_backend, deck=STARDeck())
await lh.setup()

hs = HamiltonHeaterShaker(name="hhs", index=1, interface=star_backend)
await hs.setup()

When using the STAR interface, call `setup()` only once on the `LiquidHandler`. Do not call `lh.setup()` again if it is already set up.

## Temperature control

The heater shaker exposes a {class}`~pylabrobot.capabilities.temperature_controlling.temperature_controller.TemperatureController` on `hs.tc`. For the full API, see [Temperature Control](../../capabilities/temperature-control).

In [ ]:
await hs.tc.set_temperature(37.0)
await hs.tc.wait_for_temperature()

In [ ]:
current = await hs.tc.request_temperature()
print(f"{current:.1f} \u00b0C")

The Hamilton Heater Shaker also has an edge temperature sensor, accessible via the {class}`~pylabrobot.hamilton.heater_shaker.backend.HamiltonHeaterShakerTemperatureBackend`:

In [ ]:
edge = await hs.tc.backend.request_edge_temperature()
print(f"Edge: {edge:.1f} \u00b0C")

In [ ]:
await hs.tc.deactivate()

## Shaking

The heater shaker exposes a {class}`~pylabrobot.capabilities.shaking.shaking.Shaker` on `hs.shaker`. For the full API, see [Shaking](../../capabilities/shaking).

In [ ]:
await hs.shaker.lock_plate()
await hs.shaker.shake(speed=500)
# ... do other things ...
await hs.shaker.stop_shaking()
await hs.shaker.unlock_plate()

The {class}`~pylabrobot.hamilton.heater_shaker.backend.HamiltonHeaterShakerShakerBackend` accepts `direction` (0 or 1) and `acceleration` (500--10000 rpm/s) parameters on `start_shaking`:

In [ ]:
await hs.shaker.shake(speed=800, direction=0, acceleration=2000)
await hs.shaker.stop_shaking()

## Using multiple heater shakers

### Multiple heater shakers on one control box

Each `HamiltonHeaterShaker` gets a different `index` but shares the same `HamiltonHeaterShakerBox` instance (one USB connection).

In [ ]:
from pylabrobot.hamilton.heater_shaker import HamiltonHeaterShaker, HamiltonHeaterShakerBox

box = HamiltonHeaterShakerBox()
await box.setup()

hs0 = HamiltonHeaterShaker(name="hhs0", index=0, interface=box)
hs1 = HamiltonHeaterShaker(name="hhs1", index=1, interface=box)
hs2 = HamiltonHeaterShaker(name="hhs2", index=2, interface=box)

for unit in [hs0, hs1, hs2]:
    await unit.setup()

### Two heater shakers on a STAR

The STAR has two RS232 ports. The back port is `index=1`, the front port is `index=2`.

In [ ]:
from pylabrobot.hamilton.heater_shaker import HamiltonHeaterShaker
from pylabrobot.liquid_handling import LiquidHandler, STARBackend
from pylabrobot.resources import STARDeck

star_backend = STARBackend()
lh = LiquidHandler(backend=star_backend, deck=STARDeck())
await lh.setup()

hs1 = HamiltonHeaterShaker(name="hhs1", index=1, interface=star_backend)
hs2 = HamiltonHeaterShaker(name="hhs2", index=2, interface=star_backend)

for unit in [hs1, hs2]:
    await unit.setup()

## Deck assignment

To use the heater shaker with the STAR's iSWAP or CoRe grippers (or to see it in the Visualizer), assign it to the deck. For example, using an `MFX_CAR_P3_SHAKER` carrier:

In [ ]:
from pylabrobot.resources.hamilton.mfx_carriers import MFX_CAR_P3_SHAKER

shaker_carrier = MFX_CAR_P3_SHAKER(name="shaker_carrier", modules={0: hs2, 1: hs1})
lh.deck.assign_child_resource(shaker_carrier, rails=5)

## Teardown

In [ ]:
await hs.stop()